# 91 Dashboard export

Also not analysis: this copies compact versions of the results into `outputs/tables/app/` and `outputs/tables/`, so
the dashboard runs from a fresh copy of the repository without the large intermediate files. The one regression in
here is the same "are products with a shift harder?" model as in 09a, recomputed from the saved errors so the
dashboard can show it.

In [1]:
import sys
sys.path.append('../src')
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from metrics import standardized_mean_diff

T = "../outputs/tables/"
APP = T + "app/"
os.makedirs(APP, exist_ok=True)

In [2]:
# Items: segmentation + break flags (pbi_dim_item) + ADF stationarity for the 726 tested items
items = pd.read_csv(T + "pbi_dim_item.csv")
adf = pd.concat([pd.read_csv(f"../data/interim/{c}_adf_results.csv") for c in ['foods', 'hobbies', 'household']])
items = items.merge(adf[['item_id', 'adf_stat', 'p_value', 'is_stationary']].rename(columns={'p_value': 'adf_p_value'}),
                    on='item_id', how='left')
items.to_csv(APP + "items.csv", index=False)

cat = pd.read_csv("../data/interim/cross_category_summary.csv").rename(columns={'Unnamed: 0': 'category'})
cat.to_csv(APP + "category_summary.csv", index=False)
print(cat.to_string(index=False))

 category  total_items  items_with_leading_zeros  pct_items_with_leading_zeros  ax_ay_candidates  items_tested  pct_stationary
    FOODS       1437.0                     714.0                          49.7             337.0         337.0            85.2
  HOBBIES        565.0                     303.0                          53.6             106.0         106.0            87.7
HOUSEHOLD       1047.0                     548.0                          52.3             283.0         283.0            83.4


In [3]:
# Leading-zero example: one real item before and after trimming
EXAMPLE = 'FOODS_1_004'
raw = pd.read_csv("../data/interim/scoped_sales_candidates.csv", parse_dates=['date'])
raw = raw[raw['item_id'] == EXAMPLE][['item_id', 'date', 'sales']].sort_values('date')
trimmed = pd.read_csv("../data/interim/foods_candidates.csv", parse_dates=['date'])
first_sale = trimmed.loc[trimmed['item_id'] == EXAMPLE, 'date'].min()
raw['kept_after_trim'] = raw['date'] >= first_sale
raw.to_csv(APP + "leading_zero_example.csv", index=False)
print(f"{EXAMPLE}: {(~raw['kept_after_trim']).sum()} leading zero days removed, first sale {first_sale.date()}")

FOODS_1_004: 403 leading zero days removed, first sale 2012-03-07


In [4]:
# Forecasts for the eval window (placebo and the identical seasonal-naive 'adaptive' copy excluded)
fc = pd.read_parquet("../data/processed/backtest_forecasts.parquet")
origins = sorted(fc['origin'].unique())
keep = fc['origin'].isin(origins[1:]) & (fc['variant'] != 'placebo') & \
       ~((fc['model'] == 'seasonal_naive') & (fc['variant'] == 'adaptive'))
fd = fc.loc[keep, ['item_id', 'date', 'origin', 'model', 'variant', 'actual', 'forecast', 'lo_90', 'hi_90']].copy()
fd['horizon'] = (fd['date'] - fd['origin']).dt.days
for c in ['actual', 'forecast', 'lo_90', 'hi_90']:
    fd[c] = fd[c].astype('float32').round(3)
fd.to_parquet(APP + "forecast_daily.parquet", index=False, compression='zstd')
print(len(fd), "rows")

618576 rows


In [5]:
# Inventory per item, with a DERIVED stockout-risk flag: simulated fill rate below the target service level
inv = pd.read_csv(T + "pbi_fact_inventory_kpi.csv")
inv['stockout_risk_flag'] = inv['fill_rate'] < inv['service_level']
inv.to_csv(APP + "inventory_policy.csv", index=False)
print(inv.groupby(['policy', 'service_level'])['stockout_risk_flag'].mean().round(3).unstack().to_string())

service_level           0.90   0.95   0.99
policy                                    
lightgbm_adaptive      0.079  0.148  0.314
lightgbm_static        0.071  0.136  0.311
seasonal_naive_static  0.103  0.163  0.276


In [6]:
# Selection-bias check (the same model as 09a)
err = pd.read_csv("../data/processed/backtest_errors.csv", parse_dates=['origin'])
EVAL = sorted(err['origin'].unique())[1:]
ev = err[err['origin'].isin(EVAL) & (err['variant'] == 'static') & (err['model'] != 'tsb')]
item_mase = ev.groupby(['item_id', 'category', 'model'], as_index=False)['mase'].mean()

long_df = pd.concat([pd.read_csv(f"../data/interim/{c}_candidates.csv", parse_dates=['date'])
                     for c in ['foods', 'hobbies', 'household']])
pre = long_df[long_df['date'] <= sorted(err['origin'].unique())[0]]
cov = pre.groupby('item_id')['sales'].agg(mean_sales='mean', std_sales='std', zero_share=lambda s: (s == 0).mean(),
                                         n_days='size').reset_index()
cov['cv'] = cov['std_sales'] / cov['mean_sales']
breaks = pd.read_csv("../data/processed/breaks_by_origin.csv", parse_dates=['origin'])
cov['break_item'] = cov['item_id'].map(breaks[breaks['origin'].isin(EVAL)].groupby('item_id')['has_break'].any())

rows = []
for c in ['mean_sales', 'cv', 'zero_share', 'n_days']:
    a, b = cov.loc[cov['break_item'], c], cov.loc[~cov['break_item'], c]
    rows.append({'kind': 'balance', 'variable': c, 'break_items': a.mean(), 'no_break_items': b.mean(),
                 'smd': standardized_mean_diff(a, b), 'n_break': len(a), 'n_no_break': len(b)})
reg = item_mase.merge(cov, on='item_id')
for model in ['ets', 'lightgbm']:
    d = reg[reg['model'] == model].assign(log_mase=lambda x: np.log(x['mase']), log_mean=lambda x: np.log(x['mean_sales']),
                                          break_item=lambda x: x['break_item'].astype(int))
    for name, f in [('raw', 'log_mase ~ break_item'),
                    ('adjusted', 'log_mase ~ break_item + log_mean + cv + zero_share + n_days + C(category)')]:
        m = smf.ols(f, d).fit()
        lo, hi = m.conf_int().loc['break_item']
        rows.append({'kind': 'regression', 'variable': f'{model} {name}',
                     'pct_effect': 100 * (np.exp(m.params['break_item']) - 1),
                     'ci_low_pct': 100 * (np.exp(lo) - 1), 'ci_high_pct': 100 * (np.exp(hi) - 1),
                     'p_value': m.pvalues['break_item'], 'n_break': int(d['break_item'].sum()),
                     'n_no_break': int((1 - d['break_item']).sum())})
sb = pd.DataFrame(rows)
sb.to_csv(APP + "selection_bias.csv", index=False)
print(sb.round(3).to_string(index=False))

      kind          variable  break_items  no_break_items    smd  n_break  n_no_break  pct_effect  ci_low_pct  ci_high_pct  p_value
   balance        mean_sales       14.605          16.795 -0.115      488         238         NaN         NaN          NaN      NaN
   balance                cv        0.669           0.663  0.043      488         238         NaN         NaN          NaN      NaN
   balance        zero_share        0.084           0.077  0.096      488         238         NaN         NaN          NaN      NaN
   balance            n_days     1627.074        1486.143  0.448      488         238         NaN         NaN          NaN      NaN
regression           ets raw          NaN             NaN    NaN      488         238       0.402      -3.617        4.588    0.847
regression      ets adjusted          NaN             NaN    NaN      488         238       1.627      -2.309        5.722    0.423
regression      lightgbm raw          NaN             NaN    NaN      488   

In [7]:
for f in sorted(os.listdir(APP)):
    print(f"{f:32s} {os.path.getsize(APP + f) / 1e6:6.2f} MB")

category_summary.csv               0.00 MB
forecast_daily.parquet             2.95 MB
inventory_policy.csv               3.48 MB
items.csv                          0.41 MB
leading_zero_example.csv           0.06 MB
selection_bias.csv                 0.00 MB


## Files for the one-product walkthrough on the dashboard

In [8]:
# raw and trimmed daily sales, and the weekly STL parts, for the 150 sample products and the four totals
import sys
sys.path.append("../src")
from statsmodels.tsa.seasonal import STL
from stationarity import fix_christmas

sales = pd.read_csv("../data/raw/sales_train_evaluation.csv")
cal = pd.read_csv("../data/raw/calendar.csv", parse_dates=["date"])
day_cols = [c for c in sales.columns if c.startswith("d_")]
ca_raw = sales[sales["state_id"] == "CA"].groupby("item_id")[day_cols].sum()
ca_raw.columns = cal.set_index("d").loc[day_cols, "date"]
del sales
long_df = pd.concat([pd.read_csv(f"../data/interim/{c}_candidates.csv", parse_dates=["date"])
                     for c in ["foods", "hobbies", "household"]], ignore_index=True)
long_df["category"] = long_df["item_id"].str.split("_").str[0]
totals = {cat: long_df[long_df["category"] == cat].groupby("date")["sales"].sum() for cat in ["FOODS", "HOBBIES", "HOUSEHOLD"]}
totals["CA_TOTAL"] = long_df.groupby("date")["sales"].sum()
sample = pd.read_csv("../outputs/tables/classical_sample_items.csv")

ids = sample["item_id"].tolist()
raw = ca_raw.loc[ids].stack().rename("sales_raw").reset_index()
raw.columns = ["item_id", "date", "sales_raw"]
first = long_df.groupby("item_id")["date"].min()
raw["kept_after_trim"] = raw["date"] >= raw["item_id"].map(first)
raw.to_parquet("../outputs/tables/demo_series.parquet", index=False)

parts = []
for name in ids + list(totals):
    s = totals[name] if name in totals else long_df[long_df["item_id"] == name].set_index("date")["sales"].astype(float)
    s = (fix_christmas(s) if name in totals else s).iloc[-364:]
    r = STL(s, period=7, robust=True).fit()
    parts.append(pd.DataFrame({"series": name, "date": s.index, "observed": s.to_numpy(), "trend": r.trend.to_numpy(),
                               "seasonal": r.seasonal.to_numpy(), "resid": r.resid.to_numpy()}))
pd.concat(parts).to_parquet("../outputs/tables/demo_decomposition.parquet", index=False)
print(len(raw), "raw rows,", sum(len(p) for p in parts), "decomposition rows")

291150 raw rows, 56056 decomposition rows


## What we found

Nothing new. The regression numbers match 09a, which is the check that the export reads the right files.

## What comes next

The dashboard: `python -m streamlit run dashboard/app.py`.

In [9]:
# files this notebook wrote
from pathlib import Path
ROOT = Path('..').resolve()
written = [ROOT / p for p in ['outputs/tables/app/items.csv', 'outputs/tables/app/category_summary.csv', 'outputs/tables/app/leading_zero_example.csv', 'outputs/tables/app/forecast_daily.parquet', 'outputs/tables/app/inventory_policy.csv', 'outputs/tables/app/selection_bias.csv', 'outputs/tables/demo_series.parquet', 'outputs/tables/demo_decomposition.parquet']]
for p in written:
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\app\items.csv 408 KB
outputs\tables\app\category_summary.csv 0 KB
outputs\tables\app\leading_zero_example.csv 62 KB
outputs\tables\app\forecast_daily.parquet 2,954 KB
outputs\tables\app\inventory_policy.csv 3,476 KB
outputs\tables\app\selection_bias.csv 1 KB
outputs\tables\demo_series.parquet 704 KB
outputs\tables\demo_decomposition.parquet 1,757 KB
